# Laboratoire 4 — Téléportation quantique

**Master/PhD — Informatique quantique**

Ce laboratoire explore le protocole de téléportation quantique avec QuTiP, Qiskit et Cirq. Nous vérifions la fidélité du transfert d'état et étendons le protocole au swapping d'intrication.

## 1. Rappel théorique

### Circuit de téléportation

Le protocole transfère un état inconnu $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ d'Alice à Bob en utilisant une paire intriquée $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ et deux bits classiques.

1. Préparation de l'intrication : $H$ sur Bob, $CNOT$ entre Alice et Bob
2. Alice applique $CNOT$ (contrôle : qubit $|\psi\rangle$, cible : son qubit intriqué), puis $H$
3. Mesure projective d'Alice sur ses deux qubits → 2 bits classiques
4. **Feed-forward** : Bob applique $X^a Z^b$ selon les résultats $a,b$

L'état de Bob après correction est exactement $|\psi\rangle$.

In [ ]:
import numpy as np
import qutip as qt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import cirq
import matplotlib.pyplot as plt
SNOT = (1/np.sqrt(2)) * qt.Qobj([[1,1],[1,-1]])

print('Bibliothèques chargées avec succès.')

## 2. QuTiP — Simulation du circuit avec opérateurs unitaires

Nous implémentons le protocole de téléportation en utilisant QuTiP avec des opérateurs agissant sur l'espace de Hilbert $\mathcal{H}_A \otimes \mathcal{H}_{A'} \otimes \mathcal{H}_B$.

In [ ]:
def teleportation_qutip(alpha, beta):
    """
    Simule la téléportation avec QuTiP.
    Arguments:
        alpha, beta : coefficients de l'état à téléporter
    Retourne:
        rho_bob : matrice densité de l'état reçu par Bob
    """
    N = 2  # 3 qubits : psi (0), Alice (1), Bob (2)

    # État à téléporter : |psi⟩_0
    psi = qt.Qobj(np.array([alpha, beta]))

    # État initial : |psi⟩_0 ⊗ |00⟩_{1,2}
    etat_init = qt.tensor(psi, qt.basis(N, 0), qt.basis(N, 0))

    # Intrication Alice-Bob : H_1 · CNOT_{1→2}
    H1 = qt.tensor(qt.qeye(N), SNOT, qt.qeye(N))
    CNOT12 = qt.tensor(qt.qeye(N), qt.cnot(N, N, 1, 2), qt.qeye(N))

    # Portes d'Alice : CNOT_{0→1} · H_0
    CNOT01 = qt.tensor(qt.cnot(N, N, 0, 1), qt.qeye(N))
    H0 = qt.tensor(SNOT, qt.qeye(N), qt.qeye(N))

    U_prep = CNOT12 * H1
    U_alice = H0 * CNOT01

    etat_intrication = U_prep * etat_init
    etat_apres_alice = U_alice * etat_intrication

    # Projecteurs de mesure pour Alice
    P00 = qt.tensor(qt.basis(N,0)*qt.basis(N,0).dag(), qt.basis(N,0)*qt.basis(N,0).dag(), qt.qeye(N))
    P01 = qt.tensor(qt.basis(N,0)*qt.basis(N,0).dag(), qt.basis(N,1)*qt.basis(N,1).dag(), qt.qeye(N))
    P10 = qt.tensor(qt.basis(N,1)*qt.basis(N,1).dag(), qt.basis(N,0)*qt.basis(N,0).dag(), qt.qeye(N))
    P11 = qt.tensor(qt.basis(N,1)*qt.basis(N,1).dag(), qt.basis(N,1)*qt.basis(N,1).dag(), qt.qeye(N))

    # Probabilités et états de Bob pour chaque résultat
    X = qt.tensor(qt.qeye(N), qt.qeye(N), qt.sigmax())
    Z = qt.tensor(qt.qeye(N), qt.qeye(N), qt.sigmaz())
    I = qt.tensor(qt.qeye(N), qt.qeye(N), qt.qeye(N))

    corrections = { (0,0): I, (0,1): Z, (1,0): X, (1,1): X*Z }

    print('Résultats de mesure possibles :')
    for (a,b), proj in [((0,0),P00), ((0,1),P01), ((1,0),P10), ((1,1),P11)]:
        p = (etat_apres_alice.dag() * proj * etat_apres_alice).tr()
        if p > 1e-6:
            etat_bob_brut = proj * etat_apres_alice / np.sqrt(p)
            U_corr = corrections[(a,b)]
            etat_bob_corrige = U_corr * etat_bob_brut
            rho_bob = qt.ptrace(etat_bob_corrige, [2])
            print(f'  Alice mesure ({a},{b}) — prob = {p:.3f} — Bob reçoit |ψ⟩ (fidélité ≈ {qt.fidelity(rho_bob, qt.ket2dm(psi)):.4f})')

    return rho_bob

alpha, beta = 0.8, np.sqrt(1 - 0.8**2)
rho_bob = teleportation_qutip(alpha, beta)
print('\nMatrice densité finale de Bob :')
print(rho_bob)
print('\nÉtat original |ψ⟩ :')
print(qt.ket2dm(qt.Qobj(np.array([alpha, beta]))))

## 3. Qiskit — Circuit complet avec mesures et opérations conditionnelles

Nous utilisons le système de mesure classique et les opérations conditionnelles (`if_test`) de Qiskit.

In [ ]:
def circuit_teleportation_qiskit():
    qr = QuantumRegister(3, 'q')
    cr = ClassicalRegister(3, 'c')
    qc = QuantumCircuit(qr, cr)

    # État à téléporter : |ψ⟩ = Ry(θ)|0⟩ (on peut faire varier θ)
    theta = np.pi/3
    qc.ry(theta, 0)

    # Intrication Alice-Bob
    qc.h(1)
    qc.cx(1, 2)

    # Portes d'Alice
    qc.cx(0, 1)
    qc.h(0)

    # Mesures d'Alice
    qc.measure(0, 0)
    qc.measure(1, 1)

    # Feed-forward : correction conditionnelle de Bob
    with qc.if_test((cr, 2)):
        qc.x(2)   # si cr[1]=1 (soit bit 1)
    with qc.if_test((cr, 1)):
        qc.z(2)   # si cr[0]=1

    # Mesure finale de Bob
    qc.measure(2, 2)

    return qc, theta

qc, theta = circuit_teleportation_qiskit()
qc.draw('mpl')
print(qc.draw())

sim = AerSimulator()
qc_t = transpile(qc, sim)
result = sim.run(qc_t, shots=4096).result()
counts = result.get_counts()
print('\nRésultats (c₂ c₁ c₀) :')
print(counts)

In [ ]:
# Analyse : on isole le résultat de Bob
def analyser_teleportation(counts):
    total = sum(counts.values())
    probas_bob = {}
    for bits, n in counts.items():
        bit_bob = bits[0]  # c2 (MSB)
        probas_bob[bit_bob] = probas_bob.get(bit_bob, 0) + n / total
    return probas_bob

probas_bob = analyser_teleportation(counts)
print(f'Probabilités de mesure de Bob : {probas_bob}')
print(f'État préparé par Alice : Ry({theta:.3f})|0⟩ = cos(θ/2)|0⟩ + sin(θ/2)|1⟩')
p1_attendu = np.sin(theta/2)**2
print(f'P(|1⟩) attendue sur q₀ : {p1_attendu:.4f}')
print(f'P(|1⟩) mesurée par Bob   : {probas_bob.get("1", 0):.4f}')
print(f'Correspondance : {"✓" if abs(probas_bob.get("1", 0) - p1_attendu) < 0.05 else "✗"}')

## 4. Cirq — Implémentation alternative

Nous reproduisons le protocole de téléportation avec le framework Cirq de Google.

In [ ]:
def circuit_teleportation_cirq(theta=np.pi/4):
    q0, q1, q2 = cirq.LineQubit.range(3)

    circuit = cirq.Circuit([
        # État à téléporter : Ry(θ)|0⟩
        cirq.rx(theta).on(q0),

        # Intrication Alice-Bob
        cirq.H(q1),
        cirq.CNOT(q1, q2),

        # Portes d'Alice
        cirq.CNOT(q0, q1),
        cirq.H(q0),

        # Mesures d'Alice
        cirq.measure(q0, key='m0'),
        cirq.measure(q1, key='m1'),

        # Feed-forward classique
        cirq.X(q2).with_classical_controls('m1'),
        cirq.Z(q2).with_classical_controls('m0'),

        # Mesure finale
        cirq.measure(q2, key='m2'),
    ])

    return circuit

circuit_cirq = circuit_teleportation_cirq()
print('Circuit Cirq :')
print(circuit_cirq)

sim_cirq = cirq.Simulator()
result_cirq = sim_cirq.run(circuit_cirq, repetitions=4096)
hist = result_cirq.histogram(key='m2')
print(f'\nHistogramme de Bob : {dict(hist)}')

## 5. Vérification de la fidélité pour différents états

Nous vérifions que la téléportation fonctionne pour une famille d'états $|\psi(\theta, \phi)\rangle$.

In [ ]:
def calculer_fidelite(alpha, beta):
    """Calcule la fidélité de téléportation pour un état donné."""
    N = 2
    psi = qt.Qobj(np.array([alpha, beta]))
    rho_ref = qt.ket2dm(psi)

    etat_init = qt.tensor(psi, qt.basis(N, 0), qt.basis(N, 0))
    H1 = qt.tensor(qt.qeye(N), SNOT, qt.qeye(N))
    CNOT12 = qt.tensor(qt.qeye(N), qt.cnot(N, N, 1, 2), qt.qeye(N))
    CNOT01 = qt.tensor(qt.cnot(N, N, 0, 1), qt.qeye(N))
    H0 = qt.tensor(SNOT, qt.qeye(N), qt.qeye(N))

    etat_final = H0 * CNOT01 * CNOT12 * H1 * etat_init
    rho_bob = qt.ptrace(etat_final, [2])

    return qt.fidelity(rho_bob, rho_ref)

# Test sur différents états
thetas = np.linspace(0, np.pi, 6)
phis = np.linspace(0, 2*np.pi, 5)

print('Fidélité de téléportation pour différents états:')
print(f'{"θ":>8} {"φ":>8} {"Fidélité":>10}')
print('-' * 30)
for theta in thetas:
    for phi in phis:
        alpha = np.cos(theta/2)
        beta = np.exp(1j*phi) * np.sin(theta/2)
        f = calculer_fidelite(alpha, beta)
        print(f'{theta:>8.3f} {phi:>8.3f} {f:>10.6f}')

## 6. Extension : téléportation d'un état intriqué (swapping d'intrication)

Le **swapping d'intrication** permet d'intriquer deux qubits qui n'ont jamais interagi. Le protocole combine deux paires intriquées et une téléportation.

Soient les paires $(A_1, B_1)$ et $(A_2, B_2)$ intriquées. En téléportant l'état de $B_1$ vers $A_2$, on crée une intrication entre $A_1$ et $B_2$.

In [ ]:
def entanglement_swapping_qutip():
    """
    Simule le swapping d'intrication avec QuTiP.
    États câbles : |Φ⁺⟩_{A1,B1} ⊗ |Φ⁺⟩_{A2,B2}
    On téléporte B1 → A2.
    """
    N = 2
    phi_plus = (qt.tensor(qt.basis(N,0), qt.basis(N,0)) +
                qt.tensor(qt.basis(N,1), qt.basis(N,1))).unit()

    etat_init = qt.tensor(phi_plus, phi_plus)

    # Indices : A1=0, B1=1, A2=2, B2=3
    # Intrication B1-A2
    H_B1 = qt.tensor(qt.qeye(N), SNOT, qt.qeye(N), qt.qeye(N))
    CNOT_B1A2 = qt.tensor(qt.qeye(N), qt.cnot(N, N, 1, 2), qt.qeye(N))

    # Téléportation de B1 → A2
    CNOT = qt.tensor(qt.qeye(N), qt.cnot(N, N, 1, 2), qt.qeye(N))
    H_B1_2 = qt.tensor(qt.qeye(N), SNOT, qt.qeye(N), qt.qeye(N))

    etat_apres = H_B1_2 * CNOT * CNOT_B1A2 * H_B1 * etat_init

    rho_A1B2 = qt.ptrace(etat_apres, [0, 3])

    # Vérification : la fidélité avec |Φ⁺⟩ doit être 1
    f = qt.fidelity(rho_A1B2, qt.ket2dm(phi_plus))
    print(f'Fidélité de l\'état A₁B₂ avec |Φ⁺⟩ : {f:.6f}')

    # Concurrence (mesure d'intrication)
    concur = qt.concurrence(rho_A1B2)
    print(f'Concurrence de A₁B₂ : {concur:.6f}')

    return rho_A1B2, f

rho_swap, f_swap = entanglement_swapping_qutip()
print('\nMatrice densité réduite ρ_{A1,B2} :')
print(rho_swap)

In [ ]:
def circuit_swapping_qiskit():
    """Circuit Qiskit pour le swapping d'intrication."""
    qr = QuantumRegister(4, 'q')
    cr = ClassicalRegister(4, 'c')
    qc = QuantumCircuit(qr, cr)

    # Paires intriquées : (0,1) et (2,3)
    qc.h(0)
    qc.cx(0, 1)
    qc.h(2)
    qc.cx(2, 3)

    # Téléportation du qubit 1 vers le qubit 2
    qc.cx(1, 2)
    qc.h(1)

    qc.measure(1, 1)
    qc.measure(2, 2)

    # Correction conditionnelle
    with qc.if_test((cr, 4)):
        qc.x(3)   # cr[2] = 1
    with qc.if_test((cr, 2)):
        qc.z(3)   # cr[1] = 1

    # Mesure des qubits 0 et 3 (maintenant intriqués)
    qc.measure(0, 0)
    qc.measure(3, 3)

    return qc

qc_swap = circuit_swapping_qiskit()
print('Circuit de swapping d\'intrication :')
print(qc_swap.draw())

sim = AerSimulator()
result = sim.run(transpile(qc_swap, sim), shots=4096).result()
counts = result.get_counts()
print('\nRésultats (c₃ c₂ c₁ c₀) :')
print(counts)

## 7. Questions et exercices

### Questions théoriques

1. **Pourquoi la téléportation viole-t-elle le théorème de non-clonage ?** Expliquez pourquoi l'état original d'Alice est détruit après le protocole.

2. **Rôle du feed-forward.** Que se passe-t-il si Bob applique les corrections dans le mauvais ordre ? Testez en inversant $X$ et $Z$ dans la simulation QuTiP.

3. **Mesure projective.** En utilisant les projecteurs de la partie QuTiP, montrez que les quatre résultats de mesure d'Alice sont équiprobables quel que soit $|\psi\rangle$.

4. **Téléportation sans intrication.** Pourquoi l'intrication est-elle une ressource nécessaire ? Que donne la simulation si on remplace $|\Phi^+\rangle$ par un état séparable ?

### Exercices pratiques

5. **Fidélité avec bruit.** Ajoutez un canal dépolarisant (QuTiP) sur le canal de communication entre Alice et Bob. Calculez la fidélité en fonction du taux de bruit $p$ pour $p \in [0, 1]$.

6. **Téléportation d'un état à 3 qubits.** Étendez le protocole pour téléporter l'état GHZ $|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$ en utilisant 3 paires intriquées et 3 bits classiques.

7. **Comparaison des frameworks.** Implémentez la téléportation avec les trois frameworks (QuTiP, Qiskit, Cirq) et mesurez le temps d'exécution pour 10 000 répétitions. Commentez.

8. **Swapping d'intrication avec bruit.** Ajoutez du bruit dépolarisant sur les portes du circuit de swapping et calculez la concurrence en fonction du taux de bruit. À partir de quel taux l'intrication disparaît-elle ?